# Лабораторная работа 2 — компактное решение

У 600 строк с `Ощущаемое счастье = Неизвестно` признаки состояния пустые, поэтому их нельзя напрямую подать в классификатор. Рабочая схема такая: выбрать самые связанные со счастьем состояния, восстановить их по признакам-причинам, затем классифицировать счастье по восстановленным состояниям и причинам.

Главное улучшение относительно старого `main_compact...`: состояния восстанавливаются не обычной линейной моделью, а `PolynomialFeatures(2) + Ridge`. Это добавляет умеренную нелинейность без тяжёлого кода и поднимает holdout microprecision выше 0.6.

In [1]:
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import precision_score, accuracy_score, classification_report, confusion_matrix

try:
    from IPython.display import display
except Exception:
    display = print

DATA_PATH = Path("Вар_42_Б24-215_Гаганидзе_Лаба2_студент.xlsx")
TARGET, UNKNOWN, CAT_COL = "Ощущаемое счастье", "Неизвестно", "Сфера занятости"
CLASS_ORDER = ["Hopeless", "Depressed", "Suffering", "Strugglng", "Coping", "Just ok", "Doing well", "Blooming", "Thriving", "Prospering"]
class_to_rank = {c: i for i, c in enumerate(CLASS_ORDER)}
rank_to_class = {i: c for c, i in class_to_rank.items()}

CAUSE_COLS = [
    "Среднегодовой доход, тыс. $", "Объем потребленного алкоголя в год, л.", "Количество членов семьи",
    "Количество лет образования", "Доля от дохода семьи которая тратится на продовольствие, %",
    "Коэффициент Джини сообщества", "Издержки сообщества на окружающую среду, млн. $",
    "Охват беспроводной связи в сообществе, %",
    "Количество смертей от вирусных и респираторных заболеваний в сообществе, тыс. человек",
    "Волатильность потребительских цен в сообществе",
]
STATE_COLS = [
    "Оценка благополучия", "Оценка социальной поддержки", "Ожидаемая продолжительность здоровой жизни",
    "Свобода граждан самостоятельно принимать жизненно важные решения", "Индекс Щедрости",
    "Индекс отношения к коррупции", "Оценка риска безработицы", "Индекс кредитного оптимизма",
    "Индекс страха социальных конфликтов", "Индекс семьи", "Индекс продовольственной безопасности",
    "Чувство технологического прогресса", "Чувство неравенства доходов в обществе",
]

In [2]:
df = pd.read_excel(DATA_PATH)
known = df[df[TARGET] != UNKNOWN].copy()
unknown = df[df[TARGET] == UNKNOWN].copy()
known["rank"] = known[TARGET].map(class_to_rank)

train, test = train_test_split(known, test_size=0.25, random_state=27, stratify=known["rank"])

spearman_table = pd.DataFrame([
    {"state": col, "Spearman": stats.spearmanr(train[col], train["rank"], nan_policy="omit")[0],
     "p-value": stats.spearmanr(train[col], train["rank"], nan_policy="omit")[1]}
    for col in STATE_COLS
]).assign(abs_Spearman=lambda x: x["Spearman"].abs()).sort_values("abs_Spearman", ascending=False).reset_index(drop=True)
TOP_STATES = spearman_table.head(6)["state"].tolist()

print("known / unknown:", known.shape[0], unknown.shape[0])
print("Top states:", *TOP_STATES, sep="\n- ")
display(spearman_table.round(4))

known / unknown: 1900 600
Top states:
- Оценка социальной поддержки
- Чувство технологического прогресса
- Индекс кредитного оптимизма
- Индекс продовольственной безопасности
- Индекс семьи
- Чувство неравенства доходов в обществе


,state,Spearman,p-value,abs_Spearman
0,Оценка социальной поддержки,0.7638,0.0000,0.7638
1,Чувство технологического прогресса,0.7100,0.0000,0.7100
2,Индекс кредитного оптимизма,0.6982,0.0000,0.6982
3,Индекс продовольственной безопасности,0.5984,0.0000,0.5984
4,Индекс семьи,0.5451,0.0000,0.5451
5,Чувство неравенства доходов в обществе,-0.3645,0.0000,0.3645
6,Оценка риска безработицы,-0.1777,0.0000,0.1777
7,Свобода граждан самостоятельно принимать жизне...,0.1682,0.0000,0.1682
8,Оценка благополучия,0.1180,0.0000,0.1180
9,Ожидаемая продолжительность здоровой жизни,0.0673,0.0111,0.0673


In [3]:
# МНК-часть: влияние причин на выбранные состояния.
ols_rows = []
for state in TOP_STATES:
    X, y = train[CAUSE_COLS].astype(float), train[state].astype(float)
    ols = Pipeline([("imputer", SimpleImputer()), ("scaler", StandardScaler()), ("ols", LinearRegression())]).fit(X, y)
    coefs = pd.Series(ols.named_steps["ols"].coef_, index=CAUSE_COLS)
    main_cause = coefs.abs().idxmax()
    ols_rows.append({"state": state, "R2": ols.score(X, y), "main_cause": main_cause, "std_coef": coefs[main_cause]})

display(pd.DataFrame(ols_rows).sort_values("R2", ascending=False).round(4))

,state,R2,main_cause,std_coef
0,Оценка социальной поддержки,0.9332,"Объем потребленного алкоголя в год, л.",-7.6379
3,Индекс продовольственной безопасности,0.9312,Количество смертей от вирусных и респираторных...,-12.9931
1,Чувство технологического прогресса,0.8784,"Объем потребленного алкоголя в год, л.",-6.0527
2,Индекс кредитного оптимизма,0.7356,"Издержки сообщества на окружающую среду, млн. $",23.6150
4,Индекс семьи,0.4872,"Среднегодовой доход, тыс. $",4.4186
5,Чувство неравенства доходов в обществе,0.1716,"Издержки сообщества на окружающую среду, млн. $",4.5724


In [4]:
def make_cause_X(part, columns=None):
    X = part[CAUSE_COLS].copy()
    X = X.join(pd.get_dummies(part[CAT_COL], prefix="sphere", dtype=float))
    return X if columns is None else X.reindex(columns=columns, fill_value=0)

def state_regressor():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ])

def classifier_X(pred_states, cause_X):
    Z = pd.DataFrame(pred_states, index=cause_X.index, columns=["pred_" + s for s in TOP_STATES])
    return Z.join(cause_X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_train_cause = make_cause_X(train)
X_test_cause = make_cause_X(test, X_train_cause.columns)

reg = state_regressor()
train_state_oof = cross_val_predict(reg, X_train_cause, train[TOP_STATES].astype(float), cv=cv.split(X_train_cause, train["rank"]))
reg.fit(X_train_cause, train[TOP_STATES].astype(float))
test_state_pred = reg.predict(X_test_cause)

clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(C=1.5, max_iter=5000, solver="lbfgs")),
])
clf.fit(classifier_X(train_state_oof, X_train_cause), train["rank"])
test_pred = clf.predict(classifier_X(test_state_pred, X_test_cause))

micro = precision_score(test["rank"], test_pred, average="micro", zero_division=0)
print(f"microprecision: {micro:.4f}")
print(f"accuracy:       {accuracy_score(test['rank'], test_pred):.4f}")
print(classification_report(test["rank"], test_pred, target_names=CLASS_ORDER, zero_division=0))
display(pd.DataFrame(confusion_matrix(test["rank"], test_pred, labels=range(len(CLASS_ORDER))), index=CLASS_ORDER, columns=CLASS_ORDER))

microprecision: 0.6042
accuracy:       0.6042
              precision    recall  f1-score   support

    Hopeless       0.63      0.76      0.69        54
   Depressed       0.67      0.64      0.65        55
   Suffering       0.50      0.54      0.52        69
   Strugglng       0.45      0.43      0.44        47
      Coping       0.47      0.38      0.42        66
     Just ok       0.46      0.49      0.48        51
  Doing well       0.75      0.86      0.80        66
    Blooming       0.78      0.72      0.75        39
    Thriving       0.89      0.65      0.76        26
  Prospering       1.00      1.00      1.00         2

    accuracy                           0.60       475
   macro avg       0.66      0.65      0.65       475
weighted avg       0.60      0.60      0.60       475



,Hopeless,Depressed,Suffering,Strugglng,Coping,Just ok,Doing well,Blooming,Thriving,Prospering
Hopeless,41,10,2,0,0,1,0,0,0,0
Depressed,19,35,0,0,0,1,0,0,0,0
Suffering,3,1,37,6,10,7,4,1,0,0
Strugglng,0,0,10,20,7,4,3,3,0,0
Coping,2,3,12,9,25,14,0,1,0,0
Just ok,0,3,13,2,7,25,1,0,0,0
Doing well,0,0,0,3,1,1,57,2,2,0
Blooming,0,0,0,3,3,0,5,28,0,0
Thriving,0,0,0,1,0,1,6,1,17,0
Prospering,0,0,0,0,0,0,0,0,0,2


In [5]:
# Финальное обучение на всех известных строках и прогноз 600 unknown.
X_known_cause = make_cause_X(known)
X_unknown_cause = make_cause_X(unknown, X_known_cause.columns)

final_reg = state_regressor()
known_state_oof = cross_val_predict(final_reg, X_known_cause, known[TOP_STATES].astype(float), cv=cv.split(X_known_cause, known["rank"]))
final_reg.fit(X_known_cause, known[TOP_STATES].astype(float))
unknown_state_pred = final_reg.predict(X_unknown_cause)

final_clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(C=1.5, max_iter=5000, solver="lbfgs")),
])
final_clf.fit(classifier_X(known_state_oof, X_known_cause), known["rank"])
unknown_rank = final_clf.predict(classifier_X(unknown_state_pred, X_unknown_cause))

result = unknown[["ID"] + CAUSE_COLS + [CAT_COL]].copy()
for i, state in enumerate(TOP_STATES):
    result["pred_" + state] = unknown_state_pred[:, i]
result["predicted_rank"] = unknown_rank
result["predicted_happiness"] = [rank_to_class[i] for i in unknown_rank]
result.to_excel("main_unknown_predictions.xlsx", index=False)

print("unknown rows:", len(result))
print("Saved: main_unknown_predictions.xlsx")
display(result["predicted_happiness"].value_counts().rename_axis("class").reset_index(name="n"))

unknown rows: 600
Saved: main_unknown_predictions.xlsx


,class,n
0,Suffering,104
1,Doing well,95
2,Coping,74
3,Hopeless,74
4,Just ok,67
5,Depressed,59
6,Strugglng,54
7,Blooming,43
8,Thriving,27
9,Prospering,3
